In [ ]:
import numpy as np
import lmt_sim.lmt_simulation as sim

n_lmt = 100
atom_radius = 0.0
beam_waist = 5e-3
initial_velocity_z = 0.0
detuning_hz = sim.RECOIL_FREQUENCY_HZ

m, positions, velocities, amplitudes, internal_is_ground = sim.make_atom_states(
    position_x=atom_radius,
    position_y=0.0,
    position_z=0.0,
    initial_velocity_z=initial_velocity_z,
    c0=1.0,
    c1=0.0,
)

omega_laser = 2 * np.pi * (sim.TRANSITION_FREQUENCY + detuning_hz)
squiggly_amplitudes = sim.transform_state_vector(
    m,
    amplitudes,
    internal_is_ground,
    omega_laser=omega_laser,
    t=0.0,
    z=0.0,
    vz=initial_velocity_z,
    inverse=False,
)

for i in range(n_lmt):
    laser_direction = +1 if i % 2 == 0 else -1
    m, squiggly_amplitudes, internal_is_ground, positions, velocities = sim.do_gaussian_pulse(
        m,
        squiggly_amplitudes,
        internal_is_ground,
        positions,
        velocities,
        pulse_detuning=detuning_hz,
        t_pulse=sim.T_PI,
        on_axis_rabi_freq=sim.RABI_FREQ,
        beam_waist=beam_waist,
        pulse_phase=0.0,
        k_sign=laser_direction,
        vz=initial_velocity_z,
    )

ground_prob, excited_prob = sim.calculate_ground_and_excited_probabilities(
    m, squiggly_amplitudes, internal_is_ground
)
excited_fraction = excited_prob / (ground_prob + excited_prob)
excited_fraction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lmt_sim.lmt_simulation as sim

n_lmt = 100
atom_radius = 0.0
beam_waist = 5e-3
initial_velocity_z = 0.0
base_detuning_hz = sim.RECOIL_FREQUENCY_HZ

final_detuning_offsets_hz = np.linspace(-2.0, 2.0, 201) * sim.RECOIL_FREQUENCY_HZ

m, positions, velocities, amplitudes, internal_is_ground = sim.make_atom_states(
    position_x=atom_radius,
    position_y=0.0,
    position_z=0.0,
    initial_velocity_z=initial_velocity_z,
    c0=1.0,
    c1=0.0,
)

omega_laser = 2 * np.pi * (sim.TRANSITION_FREQUENCY + base_detuning_hz)
squiggly_amplitudes = sim.transform_state_vector(
    m,
    amplitudes,
    internal_is_ground,
    omega_laser=omega_laser,
    t=0.0,
    z=0.0,
    vz=initial_velocity_z,
    inverse=False,
)

current_time = 0.0
for i in range(n_lmt - 1):
    laser_direction = +1 if i % 2 == 0 else -1
    m, squiggly_amplitudes, internal_is_ground, positions, velocities = sim.do_gaussian_pulse(
        m,
        squiggly_amplitudes,
        internal_is_ground,
        positions,
        velocities,
        pulse_detuning=base_detuning_hz,
        t_pulse=sim.T_PI,
        on_axis_rabi_freq=sim.RABI_FREQ,
        beam_waist=beam_waist,
        pulse_phase=0.0,
        k_sign=laser_direction,
        vz=initial_velocity_z,
    )
    current_time += sim.T_PI

m_saved = m.copy()
positions_saved = positions.copy()
velocities_saved = velocities.copy()
squiggly_saved = squiggly_amplitudes.copy()
internal_is_ground_saved = internal_is_ground.copy()

final_laser_direction = +1 if (n_lmt - 1) % 2 == 0 else -1

excited_scan = []
for detuning_offset_hz in final_detuning_offsets_hz:
    final_detuning_hz = base_detuning_hz + detuning_offset_hz

    m_trial = m_saved.copy()
    positions_trial = positions_saved.copy()
    velocities_trial = velocities_saved.copy()
    squiggly_trial = squiggly_saved.copy()
    internal_is_ground_trial = internal_is_ground_saved.copy()

    if final_detuning_hz != base_detuning_hz:
        (
            m_trial,
            squiggly_trial,
            internal_is_ground_trial,
            positions_trial,
            velocities_trial,
        ) = sim.change_laser_frequency_in_borde_representation(
            m_trial,
            squiggly_trial,
            internal_is_ground_trial,
            positions_trial,
            velocities_trial,
            old_detuning_hz=base_detuning_hz,
            new_detuning_hz=final_detuning_hz,
            time=current_time,
        )

    m_trial, squiggly_trial, internal_is_ground_trial, positions_trial, velocities_trial = sim.do_gaussian_pulse(
        m_trial,
        squiggly_trial,
        internal_is_ground_trial,
        positions_trial,
        velocities_trial,
        pulse_detuning=final_detuning_hz,
        t_pulse=sim.T_PI,
        on_axis_rabi_freq=sim.RABI_FREQ,
        beam_waist=beam_waist,
        pulse_phase=0.0,
        k_sign=final_laser_direction,
        vz=initial_velocity_z,
    )

    ground_prob, excited_prob = sim.calculate_ground_and_excited_probabilities(
        m_trial, squiggly_trial, internal_is_ground_trial
    )
    excited_scan.append(excited_prob / (ground_prob + excited_prob))

plt.figure(figsize=(6, 4))
plt.plot(final_detuning_offsets_hz / sim.RECOIL_FREQUENCY_HZ, excited_scan)
plt.xlabel('Final pulse detuning offset (recoil)')
plt.ylabel('Excited fraction')
plt.tight_layout()